<p align="center">
<a href="https://duckietown.com"><img src="../../assets/images/dtlogo.png" alt="Duckietown Logo" width="40%"></a>
</p>

# Représentation d'état, systèmes de coordonnées et pose

Un *état* décrit la condition de notre robot à un instant précis. Cela peut inclure des informations sur sa position ou d'autres facteurs environnementaux susceptibles de l'affecter. Lors du développement d'un algorithme robotique, il nous incombe de choisir les informations à inclure dans l'état. Autrement dit, la définition d'un état dépend de la tâche à accomplir. Idéalement, un état devrait présenter les propriétés suivantes :

1. Propriété de Markov : l'état futur est indépendant du passé étant donné l'état présent :


$$x_{t+1} = f(x_t, x_{t-1}, \dots, x_0; u_t, \dots, u_0) = f(x_t; u_t)$$


2. Une statistique minimale suffisante pour la tâche (c'est-à-dire qu'elle ne contient que les informations nécessaires à la résolution de la tâche).
3. Permet un calcul efficace
4. Généralisable

Un exemple de représentation d'état fréquemment utilisée en robotique est la *pose*. Une *pose* représente la position du robot dans l'espace ainsi que son orientation (c'est-à-dire la direction vers laquelle il est tourné). Puisque nous évoluons dans un monde 3D, nous représentons la position du robot par ses coordonnées $(x,y,z)$. L'orientation peut être représentée par ses angles de `roll` ($\theta$), `pitch` $\phi$ et `yaw` $\psi$ (également appelés angles d'Euler). Par exemple, l'angle `yaw` $\psi$ correspond à la différence angulaire entre l'axe $x$-$y$ du robot et le frame de référence.

<figure>
  <div style="text-align:center;">
  <img src="../../assets/images/representations/roll_pitch_yaw.png">
  <p>Illustration de roll, pitch, and yaw (source: https://en.wikipedia.org/wiki/Aircraft_principal_axes).</p>
  </div>
</figure>

Notez que la position et l'orientation du robot peuvent être définies soit dans le frame global (c'est-à-dire par rapport à l'origine), soit par rapport à tout autre frame. En combinant la position et l'orientation, on peut exprimer la *pose* $\bf{x}$ sous forme vectorielle/matricielle :

$$
\bf{x} = 
\begin{bmatrix}
x & y & z & \theta & \phi & \psi
\end{bmatrix}
$$

La pose $\bf{x}$ peut également être représentée sous la forme d'une matrice de transformation homogène, puisque les poses sont des membres du Matrix Lie Group appelé le groupe **S**pecial **E**uclidien $SE(3)$ :

$$
SE(3) = \{T = 
\begin{bmatrix}
R & r \\
0^T & 1
\end{bmatrix}
\in \mathrm{R}^{4 \times 4} | R \in SO(3), r \in \mathrm{R}^{3}
\},
$$

où $R$ désigne la matrice de rotation et $r$ la composante de translation. Cette matrice $T$ possède également une propriété intéressante :

$$
T^{-1} = 
\begin{bmatrix}
R^T & -R^Tr \\
0 & 1
\end{bmatrix}
$$

L'une des raisons de représenter la pose sous cette forme est qu'elle nous permet de passer facilement d'une image à l'autre (voir plus loin dans cette activité).

Bien que nous ayons abordé la question de la pose dans $SE(3)$, comme nous utilisons un robot mobile et que nous supposons un monde plat, nous pouvons simplifier la pose de notre robot à ses seules coordonnées $(x,y)$ et à son angle de yaw (ou d'orientation).  Nous allons utiliser $\theta$ pour répresenter le yaw. En effet, outre l'hypothèse que le robot se situe toujours à une altitude constante, nous pouvons également supposer sans risque que les angles de roll et de pitch sont toujours constants puisque le robot est toujours en contact avec le sol. Dans ce cas, la pose est donc simplement :

\bf{x} =
\begin{bmatrix}

x & y & \theta
\end{bmatrix}
$, ce que nous pouvons également écrire dans $SE(2)$ :

$$
SE(2) = \{T = 
\begin{bmatrix}
R & r \\
0^T & 1
\end{bmatrix}
\in \mathrm{R}^{3 \times 3} | R \in SO(2), r \in \mathrm{R}^{2}
\},
$$

où

$$
\small
R = 
\begin{bmatrix}
\cos(\theta) & -\sin(\theta) \\
\sin(\theta) & \cos(\theta) \\
\end{bmatrix}
\ \ \ \
r = 
\begin{bmatrix}
x \\
y \\
\end{bmatrix}
$$

Donc, nous avons:

$$
T = 
\begin{bmatrix}
\cos(\theta) & -\sin(\theta) & x \\
\sin(\theta) & \cos(\theta) & y \\
0 & 0 & 1
\end{bmatrix}
$$

Commençons par nous assurer de bien comprendre comment représenter une pose en réalisant l'exercice ci-dessous.

**EXEMPLE : représentation de la pose dans $SE(2)$**

<figure>
  <div style="text-align:center;">
  <img src="../../assets/images/representations/pose_exercise.png">
  </div>
</figure>

**Question:**

La figure ci-dessus illustre la position du robot (le point orange) et son orientation, l'angle étant mesuré par rapport à l'axe $x$ (on suppose que l'angle est positif si la rotation est antihoraire). Comment exprimer la pose dans $SE(2)$ ?

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Saisissez votre réponse ici
# Conseil 1 : exprimez toujours vos angles en radians !
# Conseil 2 : vous pouvez convertir des degrés en radians grâce à la fonction np.deg2rad().

theta = None

p = np.array([
    [None, None, None],
    [None, None, None],
    [None, None, None]
])

print(p)

(Vous trouverez les solutions à cette question et à d'autres dans le [fichier de solutions](../01-Representations/solutions_pose_representation.ipynb) de ce dossier. Essayez d'y répondre vous-même avant de regarder!)

# Passage d'un frame à un autre

Il est souvent utile de convertir une pose d'un frame à un autre, car plusieurs frames peuvent être utilisés pour décrire un même problème. On peut citer comme exemples la pose d'un robot par rapport à son frame (`robot frame`) ou au frame du monde (`world frame`), ou la pose d'un obstacle par rapport au capteur qui le perçoit (`sensor frame`).

Comme mentionné dans la section précédente, représenter la pose dans $SE(n)$ permet de passer facilement d'un frame à l'autre. On peut le faire en multipliant ces matrices de transformation. Par exemple, considérons deux poses différentes, $x_a$ et $x_b$. Si l'on connaît la pose $x_a$ dans le frame d'origine $o$ (c'est-à-dire ${}^ox_a$) et la pose $x_b$ dans le frame de $x_a$ (c'est-à-dire ${}^ax_b$), on peut alors calculer la pose $x_b$ par rapport au frame d'origine $o$ à l'aide de la relation suivante :

$${}^ox_b = {}^ox_a {}^ax_b,$$

où ${}^ox_b$, ${}^ox_a$ et ${}^ax_b$ sont représentés dans $SE(n)$. De même, si l'on connaît ${}^ax_o$ et ${}^ox_b$, on peut calculer ${}^ax_b$ par la formule suivante :

$${}^ax_b = {}^ax_o {}^ox_b$$

De manière générale, on peut combiner ces transformations comme suit :

$${}^ax_f = {}^ax_b {}^bx_c  {}^cx_d {}^dx_e {}^ex_f$$

De plus, il est utile de savoir que ${}^ax_b = ({}^bx_a)^{-1}$.

Voyons maintenant quelques exemples.

**EXEMPLE : Robot frame à global frame**

Votre Duckiebot est en mission de reconnaissance critique : identifier et positionner les obstacles routiers sur la carte.

Durant sa mission, votre Duckiebot utilise le GPS et connaît sa position dans le frame global (`world`). Il rencontre son premier obstacle aux coordonnées x = 2 m et y = 0,4 m, avec une orientation θ = 110°. L'obstacle se situe à 30 cm et à 50° (sens antihoraire) du Duckiebot. Où se trouve l'obstacle dans le frame global (world) ?

<figure>
  <div style="text-align:center;">
  <img src="../../assets/images/representations/moving_frame_exercise_1.png">
  <p>Les points orange et bleu représentent respectivement le robot et l'obstacle. Déterminez la position du point bleu dans le frame global (world), connaissant la pose du Duckiebot dans ce frame et celle de l'obstacle par rapport au frame du Duckiebot.</p>
  </div>
</figure>

<!-- We denote our origin frame as $o$, ... as $a$, and ... as $b$. -->

<!-- Compute $p^o_a$ and $p^b_a$, and use them to compute $p^a_b$.  -->

In [ ]:
# Exécutez ça pour initialiser le problème

# Conseil : utilisez des unités cohérentes – les mètres et les radians sont d'excellents choix
# Information : les humains comprennent mieux les degrés que les radians. Pour éviter toute confusion, entrez et sortez les valeurs en degrés, mais effectuez les calculs en radians

duckie_pos_g = np.array([2, 0.4]) # Position de Duckiebot dans le référentiel global (en mètres)
duckie_or_g = 110 # Orientation de Duckiebot dans le référentiel global (en degrés)
obstacle_dist_to_duckie = 0.3 # Distance de l'obstacle à Duckiebot
obstacle_angle = 50 # Angle de l'obstacle par rapport à Duckiebot (en degrés)

In [ ]:
# Trouvons la réponse !

# Étape 1. Convertir les degrés en radians

# Étape 2. Matrice de transformation de Duckiebot (a) à l'origine (o)

# Étape 3. Matrice de transformation de l'obstacle (b) à Duckiebot (a)

# Étape 4. Utiliser les matrices précédentes pour calculer la transformation de l'obstacle (b) à l'origine (o)

# Étape 5. Extraire l'information demandée de la matrice de transformation

# Indiquez ici votre réponse (position de l'obstacle dans le référentiel global) à la place de « None, None »

obstacle_pos_g = np.array([None, None])
print(obstacle_pos_g)

Résultat correct: [1.71809221 0.50260604]

**EXEMPLE : World frame à robot frame**

Un habitant de Duckietown, très inquiet, vous contacte :

« Le ciel nous tombe sur la tête ! »

Un morceau important du toit de son voisin s'est effondré sur la route. On vous indique sa position aux coordonnées x = 4 m et y = -1 m (dans le frame global).

Cette information est précieuse, mais avant de l'ajouter à la carte, elle doit être vérifiée. Un Duckiebot se trouve à proximité, aux coordonnées x = 3,5 m, y = -1,2 m, orienté à θ = 45°, et aura pour mission d'atteindre le morceau de toit.

Quelles sont les coordonnées de l'obstacle décrit par l'habitant inquiet par rapport au frame du Duckiebot ?

<figure>
  <div style="text-align:center;">
  <img src="../../assets/images/representations/moving_frame_exercise_2.png">
  <p>Les points orange et bleu représentent respectivement le robot et l'obstacle. Déterminez la position du point bleu dans le référentiel du robot.</p>
  </div>
</figure>

In [ ]:
### Exécutez cette cellule pour initialiser le problème
# Conseil : veillez à utiliser des unités cohérentes

duckie_pos_g = np.array([3.5, -1.2]) # Position de Duckiebot dans le frame global (mètres)
duckie_or_g = 45 # Orientation de Duckiebot dans le frame global (degrés)

obstacle_pos_g = np.array([4, -1]) # Position de l'obstacle dans le frame global (mètres)

In [ ]:
# Envoyons le Duckiebot au bon endroit !

# Convertir les degrés en radians

# Écrire la matrice de transformation du Duckiebot (a) à l'origine (o)

# Écrire la matrice de transformation de l'obstacle (b) à l'origine (o)

# Utiliser les propriétés connues des matrices de transformation et les bases de l'algèbre linéaire pour trouver la réponse

# Extraire la position de l'obstacle de la matrice de transformation

obstacle_x_a = None

obstacle_y_a = None

# Insérer la réponse ici à la place de None, None (position de l'obstacle dans le référentiel du robot)
obstacle_pos_r = np.array([obstacle_x_a, obstacle_y_a])
print(obstacle_pos_r)

Résultat correct: [ 0.49497475 -0.21213203]

**EXEMPLE: Composition des transformations**

Le Duckiebot utilisait son GPS pour se localiser, mais celui-ci est tombé en panne. La dernière donnée reçue par le Duckiebot était : $x=3$, $y=2$, $\theta=60^o$. Nous devrons continuer à estimer la pose du Duckiebot par "dead reckoning", c'est-à-dire en estimant sa pose à partir de ses mouvements. Le Duckiebot effectue les mouvements suivants :

 - avancer de 2 m, puis de 1 m vers la droite en tournant dans le sens horaire de $30^o$

 - avancer de 1 m, se déplacer de 0,3 m vers la gauche tout en tournant dans le sens antihoraire de $60^o$


 <figure>
  <div style="text-align:center;">
  <img src="../../assets/images/representations/compose_trans_exercise_3.png">
  </div>
</figure>

Quelle est la pose finale du Duckiebot ? $x_{k+2}$ dans le world frame?

In [ ]:
### Exécutez cette cellule pour initialiser le problème
# Conseil : veillez à utiliser des unités cohérentes

x_k_pos = np.array([3.0, 2.0]) # Position de Duckiebot dans le frame global à temps k (mètres)
x_k_heading = 60 # Orientation de Duckiebot dans le frame global à temps k (degrés)

u_k_pos = np.array([2.0, 1.0]) # Changement de position du robot entre k et k + 1 (dans le frame du robot)
u_k_heading = -30 # Changement de l'orientation du robot entre k et k + 1 (dans le frame du robot)

u_k1_pos = np.array([1.0, -0.3]) # Changement de position du robot entre k et k + 1 (dans le frame du robot)
u_k1_heading = 60 # Changement de l'orientation du robot entre k et k + 1 (dans le frame du robot)

In [ ]:
# Calculez la position du Duckiebot !

# Convertir les degrés en radians

# Écrire la matrice de transformation pour le Duckiebot à temps k

# Écrire la matrice de transformation pour envoyer le Duckiebot à temps k + 1

# Écrire la matrice de transformation pour envoyer le Duckiebot à temps k + 2

# Composer les transformations ensembles

# Extraire la position et orientation du Duckiebot à temps k + 2

x_k2_pos = np.array([0.0, 0.0])
x_k2_heading = 0

print(f"La position de Duckiebot à temps k + 2 est {x_k2_pos} avec orientation {x_k2_heading}")

Résultat correct: Position [4.15, 4.47224319] avec orientation 90.0

Vous pouvez maintenant passer au [notebook sur les encodeurs de roue](../02-Wheel-Encoders/wheel_encoders.ipynb). 